## Assignment 3 - DE300

Livia Fingerson and Pedro Lacombe Farina

steps:
1. Get the movies datasets we have
2. Generate movie embeddings
3. Split users in chunks
4. Generate user embeddings

# Importing dependencies

In [4]:
# pip install boto3

In [2]:
# Importing libraries
import pandas as pd
import numpy as np
from datetime import datetime
import boto3
import requests
import zipfile
import tempfile
from pathlib import Path
from botocore.exceptions import ClientError
import json

# Starting s3 session
session = boto3.Session(region_name='us-east-1')
s3 = session.client('s3')

SOURCE_BUCKET = 'pedro-lacombefarina'
BUCKET = 'fingerson-lacombefarina-mwaa'

MOVIES_PATH = 'datasets/ml-1m/movies.dat'
RATINGS_PATH = 'datasets/ml-1m/ratings.dat'
EMBEDDINGS_PATH = 'full_embedding.pt'

# Helper Functions

In [3]:
### Helper Functions
def load_ratings(BUCKET, PATH):
    '''
    This function downloads the rating file from S3 and outputs it as a pandas dataframe.
    '''
    # Creating the directory
    with tempfile.TemporaryDirectory() as td:

        td = Path(td)
        ratings_path = td / 'ratings.dat'

        # Downloading the file
        s3.download_file(BUCKET, PATH, str(ratings_path))

        # Creating the dataframe
        ratings = pd.read_csv(
            ratings_path, sep='::', engine='python', names=['user_id', 'movie_id', 'rating', 'timestamp']
        )

    return ratings

def load_movies(BUCKET, PATH):
    '''
    This function downloads the movies file from S3 and outputs it as a pandas dataframe.
    '''
    # Creating the directory
    with tempfile.TemporaryDirectory() as td:

        td = Path(td)
        movies_path = td / 'movies.dat'

        # Downloading the file
        s3.download_file(BUCKET, PATH, str(movies_path))

        # Creating the dataframe
        movies = pd.read_csv(
            movies_path, sep='::', engine='python', names=['movie_id', 'title', 'genres'], encoding='latin-1'
        )

        movie_ids = movies["movie_id"].astype(int)
        id_to_row = {int(mid): i for i, mid in enumerate(movie_ids.tolist())}

    return movies, movie_ids, id_to_row

def partition(df, n_parts=4):
    """
    Split ratings into n_parts partitions by timestamp
    (each partition has ~same number of events)
    """

    df = df.sort_values("timestamp").reset_index(drop=True)

    size = len(df) // n_parts
    df_list = []

    for i in range(n_parts):
        if i < n_parts - 1:
            df_list.append(df.iloc[i*size:(i+1)*size].copy())
        else:
            # Last partition takes remainder
            df_list.append(df.iloc[i*size:].copy())

    return df_list

# Randomly selecting user in top 5%
def pick_top_user(df, top_k=10, seed=123):
    counts = df["user_id"].value_counts()
    if len(counts) == 0:
        return None

    top_users = counts.head(top_k).index.to_numpy()
    if len(top_users) == 0:
        return None

    rng = np.random.default_rng(seed)
    return int(rng.choice(top_users))

def format_movie_list(movie_ids, items_df):
    '''
    This function formats the movie list by joining the movie id, title, and genre
    '''

    m = items_df.set_index('movie_id')[['title', 'genres']]
    parts = []
    for mid in movie_ids:
        mid = int(mid)
        if mid in m.index:
            title = m.loc[mid, 'title']
            genres = m.loc[mid, 'genres']
            parts.append(f'{mid}: {title} [{genres}]')
        else:
            parts.append(str(mid))
    return ' | '.join(parts)


# Task 1

In [4]:
# Changing CPU stability (it was not running with my kernel, so I needed to make these adjustments; AI-assisted)
import os

# Turning off tokenizer parallelism
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# Prevention of thread explosure
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['VECLIB_MAXIMUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'

In [5]:
import torch
from transformers import AutoTokenizer, AutoModel
import torch.nn.functional as F
# Initializing BERT
device = 'cuda' if torch.cuda.is_available() else 'cpu'
tok = AutoTokenizer.from_pretrained('bert-base-uncased')
bert = AutoModel.from_pretrained('bert-base-uncased').to(device)
bert.eval()

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [6]:
# Initializing encoder
@torch.no_grad()
def encode_texts(texts, batch_size=64, max_len=64):
    texts = pd.Series(texts).fillna('').astype(str).tolist()
    embs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inp = tok(batch, padding=True, truncation=True, max_length=max_len, return_tensors='pt').to(device)
        out = bert(**inp).last_hidden_state[:,0,:]  # [CLS]
        embs.append(out.cpu())
    return torch.cat(embs, dim=0)

In [7]:
def task_1():
  '''
  This function loads the movies dataset, embeds it, and uploads it to S3.
  '''

  # Loading the movies dataset
  movies, movie_ids, id_to_row = load_movies(SOURCE_BUCKET, MOVIES_PATH)

  # Generating the embeddings
  items = movies.copy()
  items['text'] = items['title'].astype(str).fillna('') + ' [SEP] ' + items['genres'].astype(str).fillna('')
  embeddings = encode_texts(items['text'])

  # Convert PyTorch tensor to NumPy
  embeddings_np = embeddings.cpu().numpy()  # shape: [num_movies, hidden_dim]

  # Uploading the embeddings to S3
  with tempfile.TemporaryDirectory() as td:
      td = Path(td)
      emb_path = td / 'full_embedding.npy'  # change extension to .npy
      np.save(emb_path, embeddings_np)
      s3.upload_file(str(emb_path), BUCKET, 'full_embedding.npy')

  print('Upload complete')

  # # Uploading the embeddings to S3
  # with tempfile.TemporaryDirectory() as td:
  #   td = Path(td)
  #   emb_path = td / 'full_embedding.pt'

  #   torch.save(embeddings, emb_path)
  #   s3.upload_file(str(emb_path), BUCKET, EMBEDDINGS_PATH)

  #   print('Upload complete')

In [8]:
# Running task 1
task_1()

Upload complete


# Task 2

In [ ]:
def task_2():
  '''
  This function downloads the ratings dataset from S3, partitions it into timeframes with the same amount of ratings, uploads them to s3, and generates the 'run_count' txt file
  '''

  # Downloading the ratings dataset
  ratings = load_ratings(SOURCE_BUCKET, RATINGS_PATH)
  ratings['ts'] = pd.to_datetime(ratings['timestamp'], unit='s')

  # Partitioning the dataset
  partitioned_ratings = partition(ratings)

  # Creating and uploading each partition as a csv to S3
  with tempfile.TemporaryDirectory() as td:
    td = Path(td)
    for idx, part in enumerate(partitioned_ratings):
      part_path = td / f"partition_{idx+1}.csv"
      part.to_csv(part_path, index=False)
      s3.upload_file(str(part_path), BUCKET, f"partition_{idx+1}.csv")

    # Creating the run_counts csv file and uploading to S3
    run_count_path = td / "run_count.json"
    run_count_path.write_text(json.dumps(1))

    s3.upload_file(str(run_count_path), BUCKET, "dags/run_count.json")

In [12]:
task_2()